# SYDE 556/750 &mdash; Practice Notebook 6
## Spatial Semantic Pointers

**Prepares you for: Test 6.**

This is the final practice notebook. It builds on the binding/unbinding machinery from PN5 and generalises it to represent *continuous quantities* (positions in space) via **fractional binding**. You will build **Spatial Semantic Pointers (SSPs)**, see the sinc-shaped similarity profile that gives them their spatial structure, and use the algebraic shift property to translate a representation without decoding it.

Sections 1--6 are `numpy` from scratch, building directly on the binding code from PN5. Section 7 then puts SSPs into spiking neurons with `nengo_spa`, where fractional binding and circular convolution come for free.

**How to use it**
- **No solutions are released.** Each part ends with an *Expected result* checkpoint so you can tell whether you are on track.
- Cells marked &#x270D; are for written answers.

**Contents**
1. Iterated binding: integer powers $B^k$
2. Fractional binding: real powers $B^x$ via the Fourier domain
3. Encoding a 1-D position as an SSP
4. Encoding a 2-D position
5. The shift property
6. Multiple objects: bundling and querying a scene
7. SSPs in Nengo: representing, shifting, and place-cell tuning
8. (Optional) Random vs hexagonal SSPs and grid cells


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(556)


# 1. Iterated binding: integer powers

In PN5 you built the binding operation $\circledast$ (circular convolution). For integer $k \ge 0$ we can define
$$B^{k} \;=\; \underbrace{B \circledast B \circledast \dots \circledast B}_{k\;\text{times}},$$
with $B^{0} = \delta$ (the identity for circular convolution, i.e. the vector $[1, 0, 0, \dots]$). This is what we used in PN5 to "count": $\mathrm{ONE} = B$, $\mathrm{TWO} = B^{2}$, $\mathrm{THREE} = B^{3}$, etc.


**a) Implement iterated binding.** Re-use your `bind` from PN5. Write a function `bind_power_int(B, k)` that returns $B^{k}$ for any integer $k \ge 0$. (For $k = 0$, return the identity vector.) Generate a random unit vector $B$ in $d = 256$ and compute $B^{k}$ for $k = 0, 1, 2, \ldots, 10$. Plot the dot product $\langle B^{k_1}, B^{k_2}\rangle$ as a heat map over $k_1, k_2 \in \{0, \ldots, 10\}$.


In [ ]:
# bind_power_int(B, k); generate B^k for k=0..10; heat-map of pairwise dot products.
# TODO


**b) What happens to $\|B^k\|$ as $k$ grows?** For a random $B$ in $d = 256$, plot $\|B^k\|$ for $k = 0, 1, \ldots, 50$. What do you observe? In one sentence, why?

&#x270D; *Your answer here.*


In [ ]:
# Plot ||B^k|| for k = 0..50.
# TODO


**c) Unitary vectors.** A vector is **unitary** if circular convolution preserves its length: $\|B \circledast B\| = \|B\|$. In the Fourier domain this is equivalent to *every* Fourier coefficient having unit magnitude, so only the phases vary. Write a function `random_unitary_vector(d, rng)` that constructs such a vector by evenly sampling random phases $\theta_k \in [0, 2\pi)$ and inverse-transforming $\exp(\mathrm i\theta)$ (taking care to enforce conjugate symmetry so the result is real). Confirm that $\|B^k\| = 1$ for $k = 0, \ldots, 50$ when $B$ is unitary.


In [ ]:
# random_unitary_vector(d, rng); check ||B^k|| stays at 1 across k.
# TODO


*Expected:* with a unitary $B$, $\|B^k\|$ stays at $1$ for all $k$.  This is why all SSPs from here on use unitary axis vectors.

# 2. Fractional binding: real powers

For a unitary vector $B$ we can extend $B^k$ from integers to *real* exponents using the Fourier identity
$$B^{x} \;=\; \mathcal{F}^{-1}\bigl\{ \mathcal{F}\{B\}^{x} \bigr\} \;=\; \mathcal{F}^{-1}\bigl\{ \mathrm e^{\mathrm i \theta\, x} \bigr\},$$
where $\theta$ is the phase angle of each Fourier coefficient of $B$. This *fractional binding* (or *fractional power*) is the key operation behind SSPs.


**a) Implement fractional binding.** Write `bind_power(B, x)` that returns $B^{x}$ for any real $x$, using the Fourier identity above. Verify that for integer $x$ it agrees with the iterated `bind_power_int` from &sect;1 (within numerical precision).


In [ ]:
# bind_power(B, x) for real x; verify agreement with bind_power_int for integer x.
# TODO


**b) Continuity in $x$.** Pick a unitary $B$ in $d = 256$. Plot the similarity $\langle B^{0}, B^{x}\rangle$ as $x$ sweeps continuously over $[-5, 5]$.


In [ ]:
# Plot <B^0, B^x> vs x in [-5, 5].
# TODO


*Expected:* the similarity is $1$ at $x = 0$ and drops smoothly on either side.

**c) The sinc similarity property.** We know that for sufficiently high $d$ and randomly-sampled axis phases,
$$\langle B^{x_1}, B^{x_2}\rangle \;\approx\; \mathrm{sinc}(x_2 - x_1).$$
Overlay $\mathrm{sinc}(x)$ on your plot from (b). In one sentence, what does this property buy us for representing continuous position?

&#x270D; *Your answer here.*


In [ ]:
# Overlay sinc(x) on your similarity curve from (b).
# TODO


# 3. Encoding a 1-D position

We can now use fractional binding to encode a continuous 1-D position $x$ as a single vector:
$$S(x) \;=\; B^{x}.$$


**a) Encode a 1-D position.** Pick a unitary axis vector $X$ in $d = 512$. For 200 query positions $q \in [-5, 5]$, encode $S(q) = X^{q}$ and plot the similarity $\langle S(0.3), S(q)\rangle$ vs. $q$.


In [ ]:
# Encode S(q) for q in [-5, 5]; plot similarity with S(0.3) vs q.
# TODO


*Expected:* a sinc-shaped curve peaked at $q = 0.3$.

**b) Effect of dimension.** Repeat (a) for $d \in \{64, 256, 1024\}$. How does the noise around the sinc shape change with $d$?

&#x270D; *Your one-sentence interpretation here.*


In [ ]:
# Repeat for several d; plot the three similarity curves on one axes.
# TODO


# 4. Encoding a 2-D position

To represent a 2-D position $(x, y)$ we use **two** independent unitary axis vectors $X$ and $Y$ and bind:
$$S(x, y) \;=\; X^{x} \circledast Y^{y}.$$


**a) Encode and decode a 2-D position.** Choose unitary $X$, $Y$ in $d = 512$. Encode the position $(2, -1)$ as $S(2, -1) = X^{2} \circledast Y^{-1}$. Then evaluate the similarity $\langle S(2, -1), S(x, y)\rangle$ over a 2-D grid $(x, y) \in [-5, 5]^{2}$, and plot the result as a heat map.


In [ ]:
# Encode S(2,-1); compute the similarity heat map over the grid; plot it.
# TODO


*Expected:* a sharp peak at $(2, -1)$ on top of a background that is approximately a product of two sincs.

**b) The decode by similarity.** A standard "decode" of an SSP is: evaluate the similarity to $S(x, y)$ over a grid and take the $(x, y)$ that maximizes the similarity. Implement this and confirm that decoding $S(2, -1)$ returns approximately $(2, -1)$. Also report the peak value of the similarity. It should be very close to $1$.


In [ ]:
# Grid-search decoder; print the recovered (x, y) and the peak similarity value.
# TODO


# 5. The shift property

A key algebraic property of SSPs is that *binding two encoded positions corresponds to adding the underlying coordinates*:
$$S(x_1, y_1) \;\circledast\; S(x_2, y_2) \;=\; S(x_1 + x_2,\; y_1 + y_2).$$
This means we can translate a representation in space without ever decoding it.


**a) Verify the shift property.** Pick a starting position $(x_1, y_1) = (1, 2)$ and a shift $(x_2, y_2) = (0.5, -0.7)$. Compute $S_1 = S(1, 2)$, $S_2 = S(0.5, -0.7)$ and $S_{\rm shifted} = S_1 \circledast S_2$. Verify by grid-decoding that $S_{\rm shifted}$ encodes the position $(1.5, 1.3)$.


In [ ]:
# Verify the shift property by binding and decoding.
# TODO


**b) Path integration sketch.** Why is the shift property useful for *path integration*? Answer in one or two sentences.

&#x270D; *Your answer here.*


# 6. Multiple objects: bundling and querying a scene

We can store several objects at several positions in one vector by **bundling** bound pairs:
$$M \;=\; \sum_{i=1}^{K} \mathrm{OBJ}_i \;\circledast\; S(x_i, y_i).$$
Then we can query for the position of a specific object by unbinding with its symbol and reading off the resulting similarity map.


**a) Build a scene.** Create a vocabulary `{CAT, DOG, BIRD}` of random unit vectors in $d = 512$ (use the same `random_unit_vector` from PN5; these need not be unitary). Place them at positions $\mathrm{CAT} \to (2, 1)$, $\mathrm{DOG} \to (-1, -2)$, $\mathrm{BIRD} \to (0, 3)$. Build the scene vector $M$ as a normalised sum of bindings.


In [ ]:
# Build M = sum of OBJ_i (*) S(x_i, y_i), normalised.
# TODO


**b) Query positions.** For each object in turn, unbind $M$ with the object's vector and plot the similarity map of the result over the same $(x, y)$ grid as in &sect;4. The position of the object should show up as a peak.


In [ ]:
# For each of CAT/DOG/BIRD, unbind from M and plot the similarity heat map.
# TODO


**c) Capacity.** What goes wrong if you try to bundle too many objects? Run the same scene-querying experiment with $K = 2, 5, 10, 20$ objects placed at random positions, and plot the *accuracy of position recovery* (e.g., distance between the true position and the argmax of the similarity map) vs. $K$. Briefly comment on the trend.

&#x270D; *Your one-sentence comment here.*


In [ ]:
# Sweep K = 2, 5, 10, 20; for each K, query each object and measure recovery accuracy.
# Plot accuracy (or recovery error) vs K.
# TODO


# 7. SSPs in Nengo

Everything so far has been `numpy`. Now we put SSPs into spiking neurons. The point of this section is to show how little extra work that takes: `nengo_spa` already knows how to do fractional binding, so an SSP is just a (unitary) semantic pointer like any other, and the network is a handful of lines.

Two things to know about the `nengo_spa` API:

- Fractional binding is the `**` operator on a `SemanticPointer`. If `X` is a suitable axis vector, `X**2.5` is $X^{2.5}$.
- `**` with a non-integer exponent only works on what `nengo_spa` calls a **positive unitary** vector: a unitary vector whose DC and Nyquist Fourier coefficients are pinned to $+1$. That is exactly your `random_unitary_vector` from &sect;1c with two of the random phases set to zero. The helper below is given to you.


In [ ]:
# Given: imports and a "positive unitary" axis-vector generator for nengo_spa.
import nengo
import nengo_spa as spa
from nengo_spa.algebras.hrr_algebra import HrrAlgebra
from nengo.utils.ensemble import tuning_curves

def positive_unitary(d, rng):
    """A unitary vector whose DC and Nyquist Fourier coefficients are +1.

    This is `random_unitary_vector` from Section 1c with two phases pinned,
    which is what nengo_spa requires for fractional (non-integer) powers.
    """
    n_coef = d // 2 + 1
    phases = rng.uniform(-np.pi, np.pi, n_coef)
    phases[0] = 0.0                      # DC coefficient -> +1
    if d % 2 == 0:
        phases[-1] = 0.0                 # Nyquist coefficient -> +1
    return np.fft.irfft(np.exp(1j * phases), n=d)


**a) An SSP vocabulary.** Using $d = 128$, build a `nengo_spa.Vocabulary` with the `HrrAlgebra`, and add two axis vectors `X` and `Y` built with `positive_unitary`. Write a helper `S(x, y)` that returns the `SemanticPointer` $X^{x} \circledast Y^{y}$. In `nengo_spa` this is just `(vocab['X']**x) * (vocab['Y']**y)`.

Check that you get the same sinc structure you built by hand in &sect;3: plot $\langle S(0,0),\, S(q, 0)\rangle$ for $q \in [-4, 4]$.

&#x1F4CC; `vocab.add('X', vector)` adds a specific vector under a name; `vocab['X']` retrieves it as a `SemanticPointer`, and `.v` gets the raw numpy array back out.

In [ ]:
# Build the vocabulary with two positive-unitary axis vectors, define S(x, y),
# and plot the similarity of S(0,0) to S(q,0) over q in [-4, 4].
# TODO


*Expected:* the same sinc-shaped curve as Section 3, with a peak of 1 at $q = 0$, decaying ripples either side.

**b) Represent an SSP in spiking neurons.** Build a `spa.Network` containing:

- a `spa.Transcode` that outputs the constant SSP $S(2, -1)$ as input, and
- a `spa.State` holding that vocabulary, with all $d$ dimensions in a single ensemble (pass `subdimensions=D`) and a modest `neurons_per_dimension`.

Connect the input into the state (`stim >> state`), probe the state's output with a synapse of about 30 ms, and run for a few hundred milliseconds. Average the last part of the probe to get the neural estimate $\hat S$, and report $\|\hat S\|$.

Then **decode the position**: reuse your grid-search decoder from &sect;4b to find the $(x, y)$ maximising $\langle \hat S, S(x, y)\rangle$, and report both the decoded position and the peak similarity.

In [ ]:
# Build the spa.Network (Transcode -> State), probe, run, average the probe,
# then grid-decode the represented position.
# TODO


*Expected:* $\|\hat S\|$ is a bit below 1 (the neurons only approximate the vector), and the decoder still recovers $(2, -1)$ essentially exactly, with a peak similarity somewhat below 1.

**c) How many neurons does it take?** Repeat (b) for several values of `neurons_per_dimension` (e.g. 5, 10, 25, 50). For each, record the decoding error (distance between the decoded and true position) and the peak similarity. Plot both against the neuron count.

Why does the *peak similarity* keep improving even once the *decoded position* has stopped changing?

In [ ]:
# Sweep neurons_per_dimension; record decode error and peak similarity; plot both.
# TODO


&#x270D; *Your answer here.*

**d) Shifting a position, in neurons.** In &sect;5 you saw that binding two SSPs adds their coordinates. Now do that shift *neurally*.

Extend your network with a second `spa.Transcode` supplying a shift SSP $S(0.5, -0.7)$, and a second `spa.State` to hold the result. The whole neural circular convolution is one line of `nengo_spa` action syntax:

```python
state * shift >> shifted
```

Start the first state at $S(1, 2)$, run the network, then grid-decode **both** states. Confirm the second one has moved to the shifted position.

&#x1F4CC; This one line builds a full circular-convolution network under the hood &mdash; hundreds of neurons computing $d$ element-wise products in the Fourier domain.

In [ ]:
# Add a shift Transcode and a second State; bind with `state * shift >> shifted`;
# decode both states and compare to the expected positions.
# TODO


*Expected:* the first state decodes to $(1, 2)$ and the second to $(1.5, 1.3)$, with a slightly lower peak similarity because the bind is itself computed in noisy spiking neurons.

**e) What do the individual neurons look like?** A population representing an SSP has a striking property: its neurons behave like **place cells**.

Build a plain `nengo.Ensemble` with $d$ dimensions, but instead of Nengo's default random encoders, set **each neuron's encoder to be an SSP** at a randomly chosen preferred position $(x_i, y_i) \in [-4, 4]^2$. Also raise the intercepts (e.g. `nengo.dists.Uniform(0.2, 0.8)`) so neurons stay silent away from their preferred location.

Then use `tuning_curves(ens, sim, inputs=...)` with your grid of SSP vectors to get each neuron's firing rate as a function of position, and plot the spatial firing map for a handful of neurons.

&#x1F4CC; `tuning_curves` evaluates the tuning analytically, so this is fast even for a large grid.

In [ ]:
# Build an ensemble whose encoders are SSPs at random preferred positions,
# compute tuning curves over the (x, y) grid, and plot several neurons' firing maps.
# TODO


*Expected:* each neuron fires in a compact blob centred on its own preferred position and is silent over most of the space, which is a place field. (With the *hexagonal* axis vectors of the next section, these blobs instead tile the plane periodically, which is what makes them grid cells.)

**f) Discussion.** Why does making the encoders SSPs produce clean place fields, whereas Nengo's default random encoders give neurons that fire almost everywhere? Tie your answer to the sinc similarity property from &sect;2.

&#x270D; *Your answer here.*

# 8. (Optional) Random vs. hexagonal SSPs

The construction above uses *random* axis vectors $X$, $Y$; these are called **RandSSPs**. A more principled choice picks the Fourier phases of the axis vectors at the vertices of an equilateral triangle (in 2-D; or the $m$-simplex in general). The resulting **HexSSPs** produce hexagonal grid-cell-like activity patterns over space and offer better representational accuracy for smooth functions (see Lecture 14 notes and Furlong et al. 2024).


**a) Construct a HexSSP.** Choose three 2-D vectors $\vec k_1, \vec k_2, \vec k_3$ pointing at angles $0$, $120^\circ$, $240^\circ$. Build a HexSSP by setting the Fourier phases of the axis vectors so that the projection of each frequency direction onto $(x, y)$ uses these three vectors. Visualise the resulting "similarity-to-origin" map $\langle S(0, 0), S(x, y)\rangle$ over a 2-D grid.


In [ ]:
# Construct a HexSSP and plot the similarity map; you should see hexagonal interference patterns.
# TODO


*Expected:* the similarity-to-origin map shows the hexagonal pattern that gives grid cells their characteristic structure.

---
*End of Practice Notebook 6.* Test 6 will assume you can do everything above by hand or from scratch, plus draw on PN1--PN5 for the underlying NEF and symbolic machinery.
